In [1]:
"""Task C2: iGRU Variant with Cross-Channel Mixing"""

import torch
import torch.nn as nn

class ChannelMixer(nn.Module):
    """
    Cross-channel mixing block to capture dependencies between variables
    
    This block learns interactions between different features at each timestep
    before passing to the temporal GRU.
    """
    
    def __init__(self, input_dim, hidden_dim=None):
        """
        Args:
            input_dim: Number of input features (d)
            hidden_dim: Internal dimension for mixing (default = input_dim)
        """
        super().__init__()
        
        if hidden_dim is None:
            hidden_dim = input_dim
        
        self.mixer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),  # Using GELU for smoother gradients
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, input_dim),
            nn.LayerNorm(input_dim)  # Stabilize training
        )
        
        self.residual = nn.Identity()
    
    def forward(self, x):
        """
        Args:
            x: Input of shape (batch_size, lookback, input_dim)
        
        Returns:
            Mixed features of same shape
        """
        # Apply channel mixing at each timestep independently
        # Reshape to (B*L, d), mix, then reshape back
        B, L, d = x.shape
        x_flat = x.view(B * L, d)
        mixed = self.mixer(x_flat)
        mixed = mixed.view(B, L, d)
        
        # Residual connection
        return mixed + x

class iGRUForecast(nn.Module):
    """
    iGRU: Improved GRU with cross-channel mixing
    
    Architecture:
    1. Channel mixing block to capture variable dependencies
    2. Standard GRU for temporal modeling
    3. Linear head for multi-step prediction
    
    Reference: Inspired by channel interaction mechanisms in modern RNNs
    """
    
    def __init__(self, input_dim, hidden_dim=64, horizon=24, out_dim=1,
                 num_layers=2, dropout=0.2, mix_hidden_dim=None):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.horizon = horizon
        self.out_dim = out_dim
        
        # Cross-channel mixing (the "i" in iGRU - improved channel modeling)
        self.channel_mixer = ChannelMixer(input_dim, mix_hidden_dim)
        
        # Temporal GRU
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Prediction head (with bottleneck for efficiency)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, horizon * out_dim)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights for better convergence"""
        for name, param in self.head.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: Input of shape (batch_size, lookback, input_dim)
        
        Returns:
            Predictions of shape (batch_size, horizon, out_dim)
        """
        # Step 1: Cross-channel mixing
        x_mixed = self.channel_mixer(x)  # (B, L, d)
        
        # Step 2: Temporal GRU
        gru_out, hidden = self.gru(x_mixed)  # (B, L, hidden)
        
        # Step 3: Take last hidden state
        last_hidden = gru_out[:, -1, :]  # (B, hidden)
        
        # Step 4: Prediction
        predictions = self.head(last_hidden)  # (B, H * out_dim)
        predictions = predictions.view(-1, self.horizon, self.out_dim)
        
        return predictions
    
    def get_params_count(self):
        return sum(p.numel() for p in self.parameters())
    
    def get_mixer_weights(self):
        """Return mixer weights for analysis"""
        return self.channel_mixer.mixer[0].weight.data.cpu().numpy()

# Test the model
if __name__ == "__main__":
    model = iGRUForecast(input_dim=7, hidden_dim=64, horizon=24)
    x = torch.randn(32, 96, 7)
    y = model(x)
    print(f"iGRU - Input: {x.shape}, Output: {y.shape}")
    print(f"Parameters: {model.get_params_count():,}")

iGRU - Input: torch.Size([32, 96, 7]), Output: torch.Size([32, 24, 1])
Parameters: 41,974
